# LSTM Model Univariate


In this section we implement the LSTM Model (Long Short-Term Memory) using the TimeSeriesDataset approach with one-hot encoding.

The LSTM (Long Short-Term Memory) Forecaster uses a stacked LSTM architecture with dropout regularization for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. The model leverages memory cells and gating mechanisms to capture long-term temporal dependencies effectively.

Key Insight: Each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific long-term patterns alongside exogenous features.

**Layer Breakdown:**

- **LSTM Layers**: 2 stacked LSTM layers with memory cells and three gates (input, forget, output)
- **Hidden Size**: 128 units per layer (default)
- **Dropout**: Applied between LSTM layers (if >1 layer) and before final output
- **Output Layer**: Single fully connected layer producing 1-step forecast
- **Input Features**: Value + GDP + CPI + Interest_Rate + Year + Month + One-Hot Encoding (1502 dims)

**Advantages**

- **Long-Term Memory**: Memory cells and gates effectively capture long-range dependencies (50+ timesteps)
- **Gradient Stability**: Gates prevent vanishing/exploding gradients better than vanilla RNN
- **Series-Specific Learning**: One-hot encoding allows model to learn unique patterns per series

**Limitations**
- **Computational Cost**: Three gates per cell require more computations than RNN/GRU
- **Sequential Processing**: Cannot parallelize across time steps like CNN/Transformer
- **Slow for Many Series**: One-hot encoding with 1502 series creates large feature vectors
- **Overfitting Risk**: Complex architecture may overfit on smaller datasets

In [ ]:
import torch
import torch.nn as nn

## Model

In [ ]:
class LSTMForecaster(nn.Module):
    """
    LSTM model for MULTIVARIATE time series forecasting.
    Architecture: LSTM -> Dropout -> LSTM -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: LSTM hidden dimension
            num_layers: Number of LSTM layers
            dropout: Dropout rate
        """
        super(LSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Take the output from the last time step
        last_output = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

## Model Results without Exogenous Features

In this section, the LSTM model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well the LSTM captures long-term temporal dependencies using only historical information and temporal context. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.33302 | 32 | 0.41710 | 128 | 0.00016 | 1 | 36.1s |
| 1 | 0.33701 | 128 | 0.32601 | 128 | 0.00298 | 2 | 35.28s |
| 2 | 0.33008 | 64 | 0.19452 | 128 | 0.00110 | 1 | 11.50s |




### Best Hyperparameters

Validation Loss: 0.33008

Parameters:
 - learning_rate: 0.00110
 - batch_size: 64
 - num_layers: 1
 - hidden_size: 128
 - dropout: 0.19452

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/lstm/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 1 Results](./img/univariate/lstm/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 1 Results](./img/univariate/lstm/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 100792.88 | 317.48 | 130.41 | 0.8193 | 70.12% |
| Fold 2 | 137464.00 | 370.76 | 130.90 | 0.7258 | 71.65% |
| Fold 3 | 55113.90 | 234.76 | 96.53 | 0.8951 | 61.23% |
| **Average** | **97790.59 ± 41529.98** | **307.67 ± 68.45** | **119.28 ± 17.63** | **0.8134 ± 0.0882** | **67.67% ± 5.48%** |



### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 6.1% ± 1.7% | 82 |
| 10-20% | 12.3% ± 3.0% | 165 |
| 20-30% | 14.4% ± 0.9% | 192 |
| 30-40% | 11.0% ± 1.9% | 147 |
| >40% | 56.2% ± 6.7% | 749 |

**Comparison with Baseline:**


The LSTM univariate model demonstrates superior performance compared to the baseline 3-month rolling average, achieving an average SMAPE of 67.67% ± 5.48% versus 72.26% ± 7.06%, representing an improvement of 4.59 percentage points. This result validates the model's capability to learn and exploit temporal dependencies in the data.


## Model Results with Exogenous Features